# Edge Client — Camera 3

Run this notebook to start Camera-3 edge client for ReID evaluation.

**Prerequisites:**
- Central server running at `http://your-server-ip:5005`
- Test video file (e.g., `test.mp4`)

## 1. Setup Environment

In [ ]:
!pip install -q torch torchvision torchreid ultralytics opencv-python requests --no-cache-dir

## 2. Clone Repository

In [ ]:
import os
import subprocess

repo_dir = "/content/ReID"

if not os.path.exists(repo_dir):
    print("[*] Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/ZahraParvin/ReID.git", repo_dir], check=True)
else:
    print(f"[*] Repository already at {repo_dir}")

os.chdir(repo_dir)
print(f"[*] Working directory: {os.getcwd()}")

## 3. Upload Video File (or use existing)

If you need to test with a video file, upload it here or provide a URL.

In [ ]:
from google.colab import files
import os

video_dir = "videos"
os.makedirs(video_dir, exist_ok=True)

# Option 1: Upload a video file
print("[*] Upload a video file for testing (or skip if using existing):")
print(f"    Supported formats: .mp4, .avi, .mov")
print(f"    Files will be saved to: {video_dir}/")
print("")
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith(('.mp4', '.avi', '.mov')):
        os.rename(filename, os.path.join(video_dir, filename))
        print(f"[✓] Saved: {video_dir}/{filename}")

# List available videos
if os.path.exists(video_dir):
    videos = [f for f in os.listdir(video_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
    if videos:
        print(f"\n[*] Available videos:")
        for v in videos:
            print(f"    - {v}")

## 4. Configure Camera-3 Edge Client

In [ ]:
# Configuration
CAMERA_ID = "Camera-3"
SERVER_URL = "http://127.0.0.1:5005/register_track"  # Change to your server IP if remote
VIDEO_SOURCE = "test.mp4"  # Change to your video file or 0 for webcam
SAVE_CROPS = False  # Set to True to save best crops for ground-truth annotation
CROPS_DIR = "crops"

print(f"[*] Configuration:")
print(f"    Camera ID: {CAMERA_ID}")
print(f"    Server URL: {SERVER_URL}")
print(f"    Video Source: {VIDEO_SOURCE}")
print(f"    Save Crops: {SAVE_CROPS}")
print(f"")
print(f"Edit the variables above to change configuration before running the client.")

## 5. Run Edge Client (Camera-3)

**Note:** This will stream video frames and send embeddings to the server.
Press Ctrl+C in the terminal to stop.

In [ ]:
import subprocess
import sys

# Build command
cmd = [
    sys.executable,
    "edge_client.py",
    "--camera-id", CAMERA_ID,
    "--video-source", VIDEO_SOURCE,
    "--server-url", SERVER_URL,
]

if SAVE_CROPS:
    cmd.append("--save-crops")
    cmd.extend(["--crops-dir", CROPS_DIR])

print(f"[*] Starting {CAMERA_ID} edge client...")
print(f"[*] Command: {' '.join(cmd)}")
print(f"")

try:
    subprocess.run(cmd, check=True)
except KeyboardInterrupt:
    print("\n[!] Client stopped by user.")
except Exception as e:
    print(f"[!] Error: {e}")

## 6. View Results (if crops were saved)

Display saved crop images for verification.

In [ ]:
import os
from PIL import Image
from IPython.display import Image as IPImage, display
import glob

if os.path.exists(CROPS_DIR):
    crops = glob.glob(os.path.join(CROPS_DIR, "*.jpg"))
    if crops:
        print(f"[*] Saved {len(crops)} crop(s)")
        for crop_path in crops[:10]:  # Display first 10
            print(f"\n{crop_path}")
            try:
                display(IPImage(crop_path))
            except Exception as e:
                print(f"  Error displaying: {e}")
    else:
        print(f"[*] No crops found in {CROPS_DIR}")
else:
    print(f"[*] Crops directory not found: {CROPS_DIR}")

## 7. Download Results

Download saved crops for local analysis.

In [ ]:
import shutil
from google.colab import files

if os.path.exists(CROPS_DIR) and glob.glob(os.path.join(CROPS_DIR, "*.jpg")):
    # Create zip archive
    zip_path = f"{CAMERA_ID}_crops"
    shutil.make_archive(zip_path, 'zip', CROPS_DIR)
    print(f"[*] Downloading {zip_path}.zip...")
    files.download(f"{zip_path}.zip")
else:
    print(f"[*] No crops to download.")